DH演習2026 · 教材 第47版 · 2026-r47 · 2026-11-11

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第47版
教材ID: 2026-r47
教材日付: 2026-11-11
活動: text_counts
環境: Colab / Jupyter
ソース: ef540f48552065735e2d78a4fb68a88ddefed01a
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第7回：戯曲全体から発言と人物への言及を数える

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

芥川龍之介の戯曲『二人小町』では、二人の女性と<ruby>黄泉<rp>（</rp><rt>よみ</rt><rp>）</rp></ruby>（死者の世界）の使が、生と死を巡って交渉します。若いときは死を免れようとし、老いて貧しくなると死を願う逆転を、暗い笑いを伴う劇として読めます。

| 登場人物 | 作品での役割 |
|---|---|
| <ruby>小野<rp>（</rp><rt>おの</rt><rp>）</rp></ruby>の<ruby>小町<rp>（</rp><rt>こまち</rt><rp>）</rp></ruby> | 死を免れようと、玉造を身代わりに勧める女性 |
| <ruby>玉造<rp>（</rp><rt>たまつくり</rt><rp>）</rp></ruby>の<ruby>小町<rp>（</rp><rt>こまち</rt><rp>）</rp></ruby> | 小野とは別人で、第2場では使と交渉する女性 |
| <ruby>黄泉<rp>（</rp><rt>よみ</rt><rp>）</rp></ruby>の<ruby>使<rp>（</rp><rt>つかい</rt><rp>）</rp></ruby> | 二人を黄泉へ迎えに来る使者 |
| <ruby>神将<rp>（</rp><rt>しんしょう</rt><rp>）</rp></ruby> | 小町を守護し、使の要求を退ける存在 |

資料は芥川龍之介『二人小町』全文（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）です。5節の人物のタグは教材で加えた注釈です。

## 1. 戯曲全体を取得する

ノートブックは、版を固定したTEIファイルを取得します。`div`は場、`sp`は一つの発言です。

In [ ]:
#@title 操作：本文を取得する
import xml.etree.ElementTree as ET

import pandas as pd
import requests

url = "https://raw.githubusercontent.com/TEI-EAJ/aozora_tei/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml"
T = "{http://www.tei-c.org/ns/1.0}"
tei = ET.fromstring(requests.get(url).content)
body = tei.findall(f".//{T}body")[0]
divs = body.findall(T + "div")
print("場の数：", len(divs))
print("発言の数：", sum(len(div.findall(T + "sp")) for div in divs))

## 2. 発言を表にする

`who`は発言者のID、`speaker`は本文に表示される発言者名です。ふりがなは`span type="rt"`に入っているので、`plain()`は読みを除いて文字をつなぎます。一行は一つの発言です。

In [ ]:
def plain(e):
    if e.tag == T + "span" and e.get("type") in ("rt", "rp"):
        return e.tail or ""
    return (e.text or "") + "".join(plain(child) for child in e) + (e.tail or "")

rows = []
for div in divs:
    for n, sp in enumerate(div.findall(T + "sp"), 1):
        speaker, *rest = sp
        rows.append({"act": div.get("n"), "n": n, "who": sp.get("who"), "speaker": speaker.text, "text": "".join(plain(e) for e in rest)})
speeches = pd.DataFrame(rows)
speeches.head(8)

## 3. 発言者を数える

`group_by`が`"speaker"`なら表示名ごと、`"who"`ならIDごとに発言を数えます。実行前に予想してください。表示名の「小町」は、一人の人物でしょうか。

二つ目の表は、`act`で選んだ場の発言を、IDと表示名の組で数えます。

In [ ]:
group_by = "speaker"
act = "一"

print(speeches[group_by].value_counts().to_dict())
speeches[speeches["act"] == act].groupby(["who", "speaker"]).size().reset_index(name="count")

## 4. 文字列検索で数える

タグを使わずに、`word`の文字列を数えます。数える範囲で件数が変わります。

- 読みを含む本文：ふりがなが名前の途中に入ると、検索に掛かりません。
- 読みを除いた本文：発言者名の表示とト書きも数えます。
- 発言の中：発言者名の表示と、発言の外のト書きを除きます。発言の途中のト書きは残ります。

`word`を「あの人」に変えると、だれを指すかは件数から分かりますか。

In [ ]:
word = "玉造の小町"

print("読みを含む本文：", "".join(body.itertext()).count(word))
print("読みを除いた本文：", plain(body).count(word))
print("発言の中：", speeches["text"].str.count(word).sum())
speeches[speeches["text"].str.contains(word)][["act", "n", "who", "text"]]

## 5. 注釈した言及を数える

文字列検索は、「あの人」がだれを指すかを区別しません。`material`は、第1場の発言14・15・19に人物のタグを付けた注釈案です。固有名には`persName`、代名詞などの参照表現には`rs type="person"`を付け、`ref="#tama"`で人物IDを指します。

`include_references`を`False`にすると`rs`を数えません。同じ3発言を`word`で検索した件数と比べてください。

In [ ]:
material = '''<body xmlns="http://www.tei-c.org/ns/1.0">
<sp xml:id="s1-sp14" who="#messenger"><speaker>使</speaker><p>いや、名前もあなたのように小町と云わなければいけないのです。</p></sp>
<sp xml:id="s1-sp15" who="#ono"><speaker>小町</speaker><p>小町！誰か小町と云う人はいなかったかしら。ああ、います。います。</p><stage>（発作的に笑い出しながら）</stage><p><persName ref="#tama">玉造の小町</persName>と云う人がいます。<rs type="person" ref="#tama">あの人</rs>を代りにつれて行って下さい。</p></sp>
<sp xml:id="s1-sp19" who="#ono"><speaker>小町</speaker><stage>（生き生きと）</stage><p>では<rs type="person" ref="#tama">あの人</rs>に行って貰って下さい。<rs type="person" ref="#tama">あの人</rs>はこの世にいるよりも、地獄に住みたいと云っています。誰も逢う人がいないものですから。</p></sp>
</body>'''

include_references = True

mentions = []
for e in ET.fromstring(material).iter():
    if e.tag == T + "persName" or (include_references and e.tag == T + "rs" and e.get("type") == "person"):
        mentions.append({"text": "".join(e.itertext()), "id": e.get("ref", "")[1:]})
print("タグの付いた言及：", pd.DataFrame(mentions)["id"].value_counts().to_dict())
print("文字列検索：", "".join(ET.fromstring(material).itertext()).count(word))
pd.DataFrame(mentions)

## 確認問題

第2場で、表示名が「小町」の発言はだれの発言ですか。3節の`act`を変えて再実行し、`answer`にその発言者のID（`who`の値）と発言の件数を入れてください。

In [ ]:
answer = {"who": "", "count": None}

In [ ]:
#@title 答えを確かめる
if not answer["who"] or answer["count"] is None:
    print("結果：不合格（`answer`に発言者のIDと件数を入れてください）")
elif answer == {"who": "#小野の小町", "count": 11}:
    print("結果：不合格（第1場の結果です。`act`を「二」に変えてください）")
elif answer["who"] != "#玉造の小町":
    print("結果：不合格（3節の二つ目の表で、`speaker`が「小町」の行の`who`を確かめてください）")
elif answer["count"] != 22:
    print("結果：不合格（同じ行の`count`を確かめてください）")
else:
    print("結果：合格")